# **Movie Recommender System Project**

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
print("numpy", np.__version__)
print("pandas", pd.__version__)


numpy 1.26.4
pandas 2.1.4


# ***Data Preprocessing***



#**Importing DataBases**

In [ ]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')
movies.head(1)
credits.head(1)


FileNotFoundError: [Errno 2] No such file or directory: '/content/tmdb_5000_movies.csv'

#**Merging the two DataBases**


In [ ]:
movies = movies.merge(credits,on='title')
movies.head(1)

movies  = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.head()


In [ ]:
nulls_per_column = movies.isnull().sum()
print(nulls_per_column)# so here we have 3 null values in ovberviwe soo make it correxxct

In [ ]:
movies.dropna(inplace=True)
# movies.duplicated().sum() we have no duplicates soo no ned

In [ ]:
movies.iloc[0].genres

#creating the helper function to fix the format['action','adventure...etc]

In [ ]:
import ast
def convert(obj):
  L = []
  for i in ast.literal_eval(obj):#used thhis here to ouput the string
    L.append(i['name'])
  return L



In [ ]:
movies['genres'] = movies['genres'].apply(convert)



In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

#fixing up the cast by taking only first 3 actors

In [ ]:
import ast
def convert1(obj):
  L = []
  counter = 0
  for i in ast.literal_eval(obj):#used thhis here to ouput the string
       if counter != 3:
          L.append(i['name'])
          counter+=1
       else:
            break
  return L



In [ ]:
movies['cast'] = movies['cast'].apply(convert1)


#fixing up crew to only have crew col  with director

In [ ]:
def fetch_director(obj):
  L = []
  for i in ast.literal_eval(obj):
    if i['job'] == 'Director':
      L.append(i['name'])
      break;

  return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

#fixing up the overviiew to coverting the overview to list

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies.head()

#fixing up the spaces bewteem them the words

In [ ]:
movies['genres']  = movies['genres'].apply(lambda x : [i.replace(" ","") for i in x])
movies['cast']  = movies['cast'].apply(lambda x : [i.replace(" ","") for i in x])
movies['keywords']  = movies['keywords'].apply(lambda x : [i.replace(" ","") for i in x])
movies['crew']  = movies['crew'].apply(lambda x : [i.replace(" ","") for i in x])


In [ ]:
movies.head()

#making all column into a single column called tags

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies.head()

#finally making the new dataframe with 3 colms we need for building model

In [ ]:
new_df = movies[['movie_id','title','tags']]
new_df

#before that conver the list to string

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:"".join(x))

In [ ]:
new_df['tags'][0]

#use stemm method to remove excess words

In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
print(f"NLTK version: {nltk.__version__}")


In [ ]:
ps

In [ ]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)


In [ ]:
new_def = new_df['tags'].apply(stem)

In [ ]:
new_df['tags'][0]

#coverting tags words into lowercase

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x : x.lower())

In [ ]:
new_df.head()

# we will use the concept of vectorization on the string using sklearn library

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv =  CountVectorizer(max_features=5000,stop_words='english')


In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()
vectors

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
similarity[0]

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distance = similarity[movie_index]
    movie_list = sorted(list(enumerate(distance)),reverse=True,key=lambda x:x[1])[1:6]

    for i in movie_list:
        print(new_df.iloc[i[0]].title)


In [ ]:
recommend('Avatar')


In [ ]:
import pickle

pickle.dump(new_df,open('movies.pkl','wb'))

In [ ]:
new_df['title'].values

In [ ]:
pickle.dump(new_df.to_dict(),open('movies_dict.pkl','wb'))

In [ ]:
pickle.dump(similarity,open('similarity.pkl','wb'))